# Numerical prototype for degenerate lake equations and vanishing viscosity# Numerical prototype for for the degenerate lake equations.

The project separates three main components:

1. **Inviscid lake equations** in vorticity-stream formulation.
2. **Viscous lake equations** in velocity formulation.
3. **Vanishing-viscosity comparison** in the weighted velocity norm \(L^2_b\).

The code is modular. Most implementation details are stored in the Python package:

```text
lake/
├── config.py
├── grid.py
├── operators.py
├── elliptic.py
├── inviscid.py
├── projection.py
├── viscous.py
├── diagnostics.py
└── plotting.py




# Cellule 2 — Markdown : modèle mathématique


## Mathematical setting

The computational domain is the unit disk:

$$
\Omega = \{(x,y): x^2+y^2<1\}.
$$

The boundary is:

$$
\partial\Omega = \{(x,y): x^2+y^2=1\}.
$$

The bathymetry is chosen as:

\[
b(x,y) = (1-r^2)^\alpha,
\qquad r=\sqrt{x^2+y^2},
\qquad \alpha=0.4.
\]

Near the boundary,

\[
1-r^2=(1-r)(1+r)\sim 2\,\mathrm{dist}(x,\partial\Omega),
\]

so

\[
b(x,y)\sim \mathrm{dist}(x,\partial\Omega)^\alpha.
\]

Thus \(b\to0\) at the boundary.

---

## Inviscid lake equations

The inviscid problem is solved in vorticity formulation:

\[
\partial_t\omega + u\cdot\nabla\omega = 0.
\]

The velocity is reconstructed from the stream function:

\[
\operatorname{div}\left(\frac1b\nabla\psi\right)=b\omega,
\]

\[
u = \frac1b\nabla^\perp\psi.
\]

We use the convention:

\[
\nabla^\perp\psi=(-\partial_y\psi,\partial_x\psi),
\]

and

\[
\operatorname{curl}u=\partial_xu_y-\partial_yu_x.
\]

Therefore,

\[
\omega=\frac{\operatorname{curl}u}{b}.
\]

The operator

\[
\operatorname{div}\left(\frac1b\nabla\psi\right)
\]

is singular because \(1/b\to+\infty\) near the boundary.

---

## Viscous lake equations

The viscous problem is treated in velocity formulation:

\[
\partial_t(bu_\mu)
+\operatorname{div}(b u_\mu\otimes u_\mu)
-2\mu \operatorname{div}\left(bD(u_\mu)+b\operatorname{div}(u_\mu)I\right)
+b\nabla p_\mu = 0,
\]

with

\[
\operatorname{div}(bu_\mu)=0.
\]

The projection step uses the degenerate operator:

\[
\operatorname{div}(b\nabla\phi)=\operatorname{div}(bu_\star),
\]

then

\[
u^{n+1}=u_\star-\nabla\phi.
\]

This operator is degenerate because \(b\to0\) near the boundary.

---

## Vanishing-viscosity comparison

The final comparison is performed only at the velocity level:

\[
\|u_\mu-u\|_{L^2_b}
=
\left(
\int_\Omega b |u_\mu-u|^2\,dx
\right)^{1/2}.
\]

In this prototype, the observed error also contains numerical effects:

- grid error;
- time-step error;
- elliptic solver error;
- projection consistency error;
- boundary-mask geometry error.

In [ ]:
from pathlib import Pathfrom pathlib np
import matplotlib.pyplot as plt

# Ensure that the project root is importable.
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lake.config import default_config, print_config_summary
from lake.grid import create_grid, print_grid_summary
from lake.operators import build_initial_fields, print_initial_diagnostics
from lake.elliptic import (
    assemble_singular_matrix,
    print_singular_matrix_summary,
    reconstruct_from_vorticity,
    reconstruction_diagnostics,
    print_reconstruction_diagnostics,
    reconstruction_sign_warning,
)
from lake.projection import (
    build_projection_data,
    print_projection_matrix_summary,
)
from lake.plotting import plot_initial_data

print("Imports completed.")
import sys



In [ ]:
cfg = default_config()

# Notebook-friendly default parameters.
# Increase N later for better accuracy, but N=96 keeps runtime reasonable.
cfg.N = 96

cfg.T_inviscid = 0.05
cfg.T_viscous = 0.05
cfg.T_limit = 0.05

cfg.mu_plot_values = [1.0e-2, 5.0e-3]
cfg.mu_values = [1.0e-2, 5.0e-3, 2.0e-3]

cfg.CFL_inviscid = 0.5
cfg.CFL_viscous = 0.2

cfg.dt_max_inviscid = 2.0e-3
cfg.dt_max_viscous = 5.0e-4

cfg.verbose = True

print_config_summary(cfg)


In [ ]:
grid = create_grid(cfg)

print_grid_summary(grid, cfg)

In [ ]:
fields = build_initial_fields(grid)

print_initial_diagnostics(fields)


In [ ]:
plot_initial_data(
    grid=grid,
    omega0=fields.omega0,
    u0=fields.u0,
)


## Initial data interpretation

The initial stream function1b\nabla^\perp\psi_0.The initial stream function \(\psi_0\) is a compactly supported bump away from the boundary.
\]

Therefore,

\[
bu_0=\nabla^\perp\psi_0,
\]

and one expects

\[
\operatorname{div}(bu_0)=0.
\]

The initial vorticity prototype is computed as

\[
\omega_0
=
\frac1b\operatorname{div}\left(\frac1b\nabla\psi_0\right).
\]

The local residuals printed above are finite-difference consistency diagnostics.  
The sparse elliptic reconstruction is checked next.
``

The initial velocity is defined by



In [ ]:
ell_data = assemble_singular_matrix(grid)

print_singular_matrix_summary(ell_data)

In [ ]:
rec0 = reconstruct_from_vorticity(
    omega=fields.omega0,
=fields.psi0,    grid=grid,
    u_reference=fields.u0,
    omega=fields.omega0,
    reconstruction=rec0,
    grid=grid,
)

print_reconstruction_diagnostics(diag_rec0)

if reconstruction_sign_warning(fields.psi0, rec0.psi, grid):
    print("WARNING: possible sign convention issue.")
else:
    print("Sign convention check passed.")
``
    elliptic_data=ell_data,
    cfg=cfg,
)

diag_rec0 = reconstruction_diagnostics(


In [ ]:
proj_data = build_projection_data(grid, cfg)proj_data = build_projection_data_projection_matrix_summary(proj_data)

In [ ]:
## End of Part 1

At this stage, the notebook configuration;At this stage, the notebook has constructed:
- the masked Cartesian disk grid;
- the degenerate bathymetry \(b\);
- the initial stream function \(\psi_0\);
- the initial velocity \(u_0\);
- the initial vorticity prototype \(\omega_0\);
- the singular elliptic operator for inviscid reconstruction;
- the degenerate projection operator for the viscous solver.

Next parts:

1. Run the inviscid vorticity solver.
2. Run the viscous velocity solver.
3. Compare \(u_\mu\) and \(u\) in the weighted norm \(L^2_b\).


